# 1. Project Background



# 2. Data Processing Workflow

# 3. Exploratory Analysis

# 4. Modelling Framework


In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import statsmodels.formula.api as smf
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy.stats import f_oneway
from dateutil.relativedelta import relativedelta
import warnings
import dataframe_image as dfi
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
warnings.filterwarnings('ignore')


# 5. Modelling

## 5.1 Train-Validation split

In [ ]:

from sklearn.model_selection import train_test_split

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Original version - no encoding (for catboost)
X_train_org = X_train.copy()
X_val_org = X_val.copy()


## 5.2 Linear Models


In [1]:

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler


In [ ]:
# For log-models (OLS, Ridge, Lasso, Enet)
X_train_mod = X_train.copy()
X_val_mod = X_val.copy()

# OLS
# 1. Fit OLS on log1p-transformed target 
ols = LinearRegression().fit(X_train_mod, y_train)

# 2. Predict on validation set (log scale)
y_pred_val_log = ols.predict(X_val_mod)

# 3. Evaluate directly on log scale 
rmse_val_ols = np.sqrt(mean_squared_error(y_val, y_pred_val_log))
mae_val_ols  = mean_absolute_error(y_val, y_pred_val_log)
r2_val_ols   = r2_score(y_val, y_pred_val_log)

print(f"OLS (log-scale): RMSE = {rmse_val_ols:.4f} | "
      f"MAE = {mae_val_ols:.4f} | R² = {r2_val_ols:.3f}")

# RIDGE 
#  1. Fit Ridge on log1p-transformed target 
alphas = np.logspace(-3, 3, 200)  # finer grid for better tuning
ridge = RidgeCV(alphas=alphas, cv=5, scoring='r2')
ridge.fit(X_train_mod, y_train)

#  2. Predict on validation set (log scale) 
y_pred_val_log_ridge = ridge.predict(X_val_mod)

#  3. Evaluate directly on log scale 
rmse_val_ridge = np.sqrt(mean_squared_error(y_val, y_pred_val_log_ridge))
mae_val_ridge  = mean_absolute_error(y_val, y_pred_val_log_ridge)
r2_val_ridge   = r2_score(y_val, y_pred_val_log_ridge)

print(f"Ridge (log-scale CV): RMSE = {rmse_val_ridge:.4f} | "
      f"MAE = {mae_val_ridge:.4f} | R² = {r2_val_ridge:.3f}")

# LASSO
#  1. Fit Lasso on log1p-transformed target 
lasso = LassoCV(cv=5, random_state=42, n_alphas=200)
lasso.fit(X_train_mod, y_train)

#  2. Predict on validation set (log scale) 
y_pred_val_log = lasso.predict(X_val_mod)

#  3. Evaluate directly on LOG scale 
rmse_val_lasso = np.sqrt(mean_squared_error(y_val, y_pred_val_log))
mae_val_lasso  = mean_absolute_error(y_val, y_pred_val_log)
r2_val_lasso   = r2_score(y_val, y_pred_val_log)

print(f"Lasso (log-scale CV): RMSE = {rmse_val_lasso:.4f} | "
      f"MAE = {mae_val_lasso:.4f} | R² = {r2_val_lasso:.3f}")

# ELASTIC NET
#  1. Fit Elastic Net on log1p-transformed target 
enet = ElasticNetCV(
    l1_ratio=[0.2, 0.4, 0.6, 0.8, 0.95],
    n_alphas=300,
    cv=5,
    random_state=42,
    max_iter=5000
)
enet.fit(X_train_mod, y_train)

#  2. Predict on validation set (log scale) 
y_pred_val_log = enet.predict(X_val_mod)

#  3. Evaluate directly on log scale 
rmse_val_enet = np.sqrt(mean_squared_error(y_val, y_pred_val_log))
mae_val_enet  = mean_absolute_error(y_val, y_pred_val_log)
r2_val_enet   = r2_score(y_val, y_pred_val_log)

print(f"ElasticNet (log-scale CV): RMSE = {rmse_val_enet:.4f} | "
      f"MAE = {mae_val_enet:.4f} | R² = {r2_val_enet:.3f}")


## 5.3 KNN

## 5.4 Neural Network

Adapted and refined from the project baseline implementation. 
Improvements include parameter tuning and integration into the ensemble pipeline

In [ ]:
import os

# Environment variables for deterministic behavior
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

import random
import numpy as np
import tensorflow as tf

# Set global random seeds
random.seed(42)
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

# Neural Network (Keras) model using same train/val split
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt  # visuals

# Train on log1p target to match tree models (set False if your y is already log1p)
TARGET_IS_LOG1P = True
if not TARGET_IS_LOG1P:
    y_train = np.log1p(y_train)
    y_val   = np.log1p(y_val)

# Split columns by dtype for preprocessing
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Preprocessing: impute+scale numeric, impute+one-hot categorical
num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preproc = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
], remainder="drop")

In [ ]:
#  Ensure boolean columns are numeric (0/1) 
bool_cols = X_train.select_dtypes(include=["bool"]).columns
X_train_nn = X_train.copy()
X_val_nn   = X_val.copy()

for c in bool_cols:
    X_train_nn[c] = X_train_nn[c].astype(int)
    X_val_nn[c]   = X_val_nn[c].astype(int)

In [ ]:
# Build a simple MLP for regression on log scale
def build_nn(input_dim: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    outputs = tf.keras.layers.Dense(1, activation="linear")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",  # log-scale MSE
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model

# Fit preprocessor on train only; transform both splits
X_train_nn = preproc.fit_transform(X_train_nn)
X_val_nn   = preproc.transform(X_val_nn)

# Save preproc and model for reuse
import joblib
joblib.dump(preproc, "preprocessor_nn.pkl")

# Short data summary
print("\n[NN] dtypes (train):")
print(X_train.dtypes.value_counts())
print(f"[NN] Shapes | X_train: {X_train.shape}  X_val: {X_val.shape}")
print(f"[NN] Transformed | X_train_nn: {X_train_nn.shape}  X_val_nn: {X_val_nn.shape}")


In [ ]:
# Build and train model with early stopping + LR schedule
nn_model = build_nn(X_train_nn.shape[1])
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
reduce_lr  = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5)
nn_history = nn_model.fit(
    X_train_nn, y_train,
    validation_data=(X_val_nn, y_val),
    epochs=300, batch_size=512, verbose=0,
    callbacks=[early_stop, reduce_lr],
    shuffle = False
)

# Predictions on log scale
y_pred_train_log = nn_model.predict(X_train_nn, verbose=0).ravel()
y_pred_val_log_nn   = nn_model.predict(X_val_nn,   verbose=0).ravel()

# Log-scale metrics
rmse_train_log = np.sqrt(mean_squared_error(y_train, y_pred_train_log))
rmse_val_log_nn   = np.sqrt(mean_squared_error(y_val,   y_pred_val_log_nn))
mae_train_log  = mean_absolute_error(y_train, y_pred_train_log)
mae_val_log_nn    = mean_absolute_error(y_val,   y_pred_val_log_nn)
print("\n" + "="*60)
print("Neural Network Evaluation (Log Scale)")
print("="*60)
print("Training Set:")
print(f"  RMSE: {rmse_train_log:8.4f}")
print(f"  MAE : {mae_train_log:8.4f}")
print("-"*60)
print("Validation Set:")
print(f"  RMSE: {rmse_val_log_nn:8.4f}")
print(f"  MAE : {mae_val_log_nn:8.4f}")
print("="*60 + "\n")

In [ ]:
# Key Visuals

# 1) Loss curve
plt.figure(figsize=(7,4))
plt.plot(nn_history.history["loss"], label="train_loss")
plt.plot(nn_history.history["val_loss"], label="val_loss")
plt.title("NN Training — Loss (log-scale RMSE)")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout(); plt.show()

# 2) Predicted vs Actual (log scale)
plt.figure(figsize=(5,5))
plt.scatter(y_val, y_pred_val_log_nn, s=10, alpha=0.6)
m = float(np.max([y_val.max(), y_pred_val_log_nn.max()]))
plt.plot([0, m], [0, m], 'r--')
plt.title("Predicted vs Actual (Validation)")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.tight_layout(); plt.show()

# 3) Residual histogram
resid = y_val - y_pred_val_log_nn
plt.figure(figsize=(7,4))
plt.hist(resid, bins=40, color='steelblue', edgecolor='black', alpha=0.7)
plt.title("Residual Distribution (Validation)")
plt.xlabel("Residual"); plt.ylabel("Count"); plt.tight_layout(); plt.show()


## 5.5 Random Forest

## 5.6 Decision Tree

### 5.6.1 Categorical Tree

### 5.6.2 Regression Tree

## 5.7 XGBoost

### 5.7.1 Hyperparameter Tuned XGBoost

## 5.8 CatBoost

## 5.9 Model Comparison and Ensemble Integration

In [ ]:
# Combine all model metrics 
summary_df = pd.DataFrame([
    {"Model": "OLS",         "Val_RMSE": rmse_val_ols,   "Val_MAE": mae_val_ols,   "Val_R²": r2_val_ols},
    {"Model": "Ridge",       "Val_RMSE": rmse_val_ridge, "Val_MAE": mae_val_ridge, "Val_R²": r2_val_ridge},
    {"Model": "Lasso",       "Val_RMSE": rmse_val_lasso, "Val_MAE": mae_val_lasso, "Val_R²": r2_val_lasso},
    {"Model": "ElasticNet",  "Val_RMSE": rmse_val_enet,  "Val_MAE": mae_val_enet,  "Val_R²": r2_val_enet},
    {"Model": "KNN",         "Val_RMSE": rmse_val_knn,          "Val_MAE": mae_val_knn,          "Val_R²": r2_val_knn},
    {"Model": "Random Forest (base)", "Val_RMSE": rmse_val_rf_base, "Val_MAE": mae_val_rf_base, "Val_R²": None},
    {"Model": "Random Forest (CV Tuned)", "Val_RMSE": rmse_val_rf_best, "Val_MAE": mae_val_rf_best, "Val_R²": r2_val_rf_best},
    {"Model": "Cat Tree",    "Val_RMSE": rmse_val_catt,  "Val_MAE": mae_val_catt,  "Val_R²": r2_val_catt},
    {"Model": "Regression Tree", "Val_RMSE": rmse_val_regt, "Val_MAE": mae_val_regt, "Val_R²": r2_val_regt},
    {"Model": "Neural Net",  "Val_RMSE": rmse_val_log_nn,    "Val_MAE": mae_val_log_nn,    "Val_R²": None},
    {"Model": "XGBoost (CV Tuned)", "Val_RMSE": rmse_xgb, "Val_MAE": mae_xgb,  "Val_R²": r2_xgb},
    {"Model": "CatBoost (CV Tuned)", "Val_RMSE": rmse_cb, "Val_MAE": mae_cb, "Val_R²": r2_cb}
])

# Format 
summary_df = summary_df.round(4)
summary_df = summary_df.sort_values("Val_RMSE").reset_index(drop=True)

# Display 
print("\n Overall Model Comparison Summary (Validation, Log Scale)")
print(summary_df.to_string(index=False))


## 5.10 Stacking Models

## 5.11 Equal Weight

In [ ]:
#  Align all predictions and y_val by index 
preds_dict = {
    "ridge": pd.Series(y_pred_val_log_ridge).reset_index(drop=True),
    "xgb":   pd.Series(y_pred_xgb).reset_index(drop=True),
    "catb":  pd.Series(y_pred_cb).reset_index(drop=True),
    "nn":    pd.Series(y_pred_val_log_nn).reset_index(drop=True)
}

# Ensure same length as y_val
min_len = min(len(y_val), *[len(v) for v in preds_dict.values()])
y_val_eq = y_val.iloc[:min_len].reset_index(drop=True)

# Truncate predictions to the same length
for k in preds_dict:
    preds_dict[k] = preds_dict[k].iloc[:min_len]

# Build ensemble DataFrame safely 
preds_val = pd.DataFrame(preds_dict)
print(preds_val.shape, "aligned with y_val:", len(y_val_eq))

# Equal weights (sum to 1)
w_equal = np.ones(preds_val.shape[1]) / preds_val.shape[1]

# Equal-weight ensemble prediction (log scale)
y_pred_val_eq = preds_val.values @ w_equal

# Evaluate
rmse_val_eq = np.sqrt(mean_squared_error(y_val_eq, y_pred_val_eq))
mae_val_eq  = mean_absolute_error(y_val_eq, y_pred_val_eq)
r2_val_eq   = r2_score(y_val_eq, y_pred_val_eq)

print("\n Equal-Weight Ensemble (Log Scale) ")
print(f"RMSE: {rmse_val_eq:.4f} | MAE: {mae_val_eq:.4f} | R²: {r2_val_eq:.3f}")

## 5.12 Conditional Optimal Weight (COW)

In [ ]:
preds_dict = {
    "ridge": pd.Series(y_pred_val_log_ridge).reset_index(drop=True),
    "rf":   pd.Series(y_pred_val_rf_best).reset_index(drop=True),
    "catb":  pd.Series(y_pred_cb).reset_index(drop=True),
    "nn":    pd.Series(y_pred_val_log_nn).reset_index(drop=True)
}

preds_val = pd.DataFrame(preds_dict)

assert preds_val.shape[0] == len(y_val), "Mismatch between prediction and target length."

# Compute residuals for each model
residuals = preds_val.sub(y_val.values, axis=0)
cov_matrix = residuals.cov().values

# Calculate COW weights: w = Σ⁻¹1 / (1'Σ⁻¹1)
inv_cov = np.linalg.pinv(cov_matrix)
ones = np.ones(cov_matrix.shape[0])
weights_cow = inv_cov @ ones
weights_cow /= ones.T @ inv_cov @ ones  # normalisation

# Generate ensemble predictions
y_pred_val_cow = preds_val.values @ weights_cow

# Evaluate model performance
rmse_val_cow = np.sqrt(mean_squared_error(y_val, y_pred_val_cow))
mae_val_cow = mean_absolute_error(y_val, y_pred_val_cow)
r2_val_cow = r2_score(y_val, y_pred_val_cow)

# Display results
print("Conditional Optimal Weighting (COW) Ensemble (Log Scale)")
print("Model Weights:", dict(zip(preds_val.columns, np.round(weights_cow, 3))))
print(f"Validation → RMSE: {rmse_val_cow:.4f} | MAE: {mae_val_cow:.4f} | R²: {r2_val_cow:.3f}")

## 5.13 Final Model Summary

In [ ]:
#  Combine all model metrics 
summary_df = pd.DataFrame([
    {"Model": "OLS",         "Val_RMSE": rmse_val_ols,   "Val_MAE": mae_val_ols,   "Val_R²": r2_val_ols},
    {"Model": "Ridge",       "Val_RMSE": rmse_val_ridge, "Val_MAE": mae_val_ridge, "Val_R²": r2_val_ridge},
    {"Model": "Lasso",       "Val_RMSE": rmse_val_lasso, "Val_MAE": mae_val_lasso, "Val_R²": r2_val_lasso},
    {"Model": "ElasticNet",  "Val_RMSE": rmse_val_enet,  "Val_MAE": mae_val_enet,  "Val_R²": r2_val_enet},
    {"Model": "KNN",         "Val_RMSE": rmse_val_knn,          "Val_MAE": mae_val_knn,          "Val_R²": r2_val_knn},
    {"Model": "Random Forest (Base)", "Val_RMSE": rmse_val_rf_base, "Val_MAE": mae_val_rf_base, "Val_R²": None},
    {"Model": "Random Forest (CV Tuned)", "Val_RMSE": rmse_val_rf_best, "Val_MAE": mae_val_rf_best, "Val_R²": r2_val_rf_best},
    {"Model": "Cat Tree",    "Val_RMSE": rmse_val_catt,  "Val_MAE": mae_val_catt,  "Val_R²": r2_val_catt},
    {"Model": "Regression Tree", "Val_RMSE": rmse_val_regt, "Val_MAE": mae_val_regt, "Val_R²": r2_val_regt},
    {"Model": "Neural Net",  "Val_RMSE": rmse_val_log_nn,    "Val_MAE": mae_val_log_nn,    "Val_R²": None},
    {"Model": "XGBoost (CV Tuned)", "Val_RMSE": rmse_xgb, "Val_MAE": mae_xgb,  "Val_R²": r2_xgb},
    {"Model": "CatBoost (CV Tuned)", "Val_RMSE": rmse_cb, "Val_MAE": mae_cb, "Val_R²": r2_cb},
    {"Model": "Equal Weight", "Val_RMSE": rmse_val_eq, "Val_MAE": mae_val_eq, "Val_R²": r2_val_eq},
    {"Model": "COW", "Val_RMSE": rmse_val_cow, "Val_MAE": mae_val_cow, "Val_R²": r2_val_cow},
    {"Model": "Stacking (Balanced)", "Val_RMSE": rmse_val_stack, "Val_MAE": mae_val_stack, "Val_R²": r2_val_stack},
    {"Model": "Stacking (Performance)", "Val_RMSE": rmse_val_stack2, "Val_MAE": mae_val_stack2, "Val_R²": r2_val_stack2},

])

#  Format 
summary_df = summary_df.round(4)
summary_df = summary_df.sort_values("Val_RMSE").reset_index(drop=True)

#  Display 
print("\n Overall Model Comparison Summary (Validation, Log Scale) ")
print(summary_df.to_string(index=False))

# 6. Test data

## 6.1 Data Cleaning

## 6.2 Join Mosaic

## 6.3 Feature Engineering

## 6.4 Final Models

### 6.4.1 Final Equal Weight


In [ ]:
preds_test_dict = {
    "ridge": pd.Series(y_pred_test_ridge).reset_index(drop=True),
    "xgb":   pd.Series(y_pred_test_xgb).reset_index(drop=True),
    "catb":  pd.Series(y_pred_test_cb).reset_index(drop=True),
    "nn":    pd.Series(y_pred_test_nn).reset_index(drop=True)
}

# Align and truncate in case of minor length mismatches
min_len_test = min([len(v) for v in preds_test_dict.values()])
for k in preds_test_dict:
    preds_test_dict[k] = preds_test_dict[k].iloc[:min_len_test]

preds_test = pd.DataFrame(preds_test_dict)
print(preds_test.shape, "test predictions aligned")

#  Equal-weight ensemble (sum of weights = 1) 
w_equal = np.ones(preds_test.shape[1]) / preds_test.shape[1]
y_pred_test_log_eq = preds_test.values @ w_equal

#  Convert back from log1p scale to original 
y_pred_test_eq = np.expm1(y_pred_test_log_eq)

#  Attach SupporterID for export 
test_pred_eq = pd.DataFrame({
    "SupporterID": test_features_full_df["SupporterID"].values[:len(y_pred_test_LTV_cow)],
    "pred_log_LTV": y_pred_test_log_eq,
    "pred_LTV": y_pred_test_eq
})

print("Equal-weight test predictions generated.")
print(test_pred_eq.head())

### 6.4.2 Final Conditional Optimal Weight (COW)

In [ ]:
# Fitted preprocessor (for each model’s input)
pre  # ColumnTransformer

# Optimal weights from previous COW (log-scale)
w_cow = {"xgb": 0.427, "catb": 0.401, "lgbm": 0.079, "nn": 0.092}


In [ ]:
# Combine predictions using COW weights 
preds_dict = {
    "xgb":   y_pred_test_xgb,
    "catb":  y_pred_test_cb,
    "lgbm":  y_pred_test_lgbm,
    "nn":    y_pred_test_nn
}

# Align lengths
min_len = min(len(v) for v in preds_dict.values())
preds_val = pd.DataFrame({k: v[:min_len] for k, v in preds_dict.items()})

# Weighted average in log space
y_pred_test_log_cow = (
    preds_val["xgb"]  * w_cow["xgb"]  +
    preds_val["catb"] * w_cow["catb"] +
    preds_val["lgbm"] * w_cow["lgbm"] +
    preds_val["nn"]   * w_cow["nn"]
)

# Back-transform to original $ scale 
y_pred_test_LTV_cow = np.expm1(y_pred_test_log_cow)

# Export for submission 
test_pred_cow = pd.DataFrame({
    "SupporterID": test_features_full_df["SupporterID"].values[:len(y_pred_test_LTV_cow)],
    "pred_log_LTV": y_pred_test_log_cow,
    "pred_LTV": y_pred_test_LTV_cow
})
test_pred_cow.to_csv("TestPred_COW.csv", index=False)

print("\nSaved: TestPred_COW.csv")
print(f"Mean predicted LTV: {y_pred_test_LTV_cow.mean():,.2f}")
print(f"Median predicted LTV: {np.median(y_pred_test_LTV_cow):,.2f}")